### Legacy access 

In [0]:
dbutils.widgets.text("storage_account", "")
dbutils.widgets.text("container", "")
dbutils.widgets.text("secret_scope", "")

storage_account = dbutils.widgets.get("storage_account")
container       = dbutils.widgets.get("container")
scope           = dbutils.widgets.get("secret_scope")

assert all([storage_account, container, scope])

### Canonical SPN OAuth config (docs pattern) creds read from the secret scope.

In [0]:
configs = {
    "fs.azure.account.auth.type": "OAuth",
    "fs.azure.account.oauth.provider.type": "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider",
    "fs.azure.account.oauth2.client.id": dbutils.secrets.get(scope, "sp-databricks-adls-appid"),
    "fs.azure.account.oauth2.client.secret": dbutils.secrets.get(scope, "sp-databricks-adls-appkey"),
    "fs.azure.account.oauth2.client.endpoint":
        f"https://login.microsoftonline.com/{dbutils.secrets.get(scope, 'tenant-id')}/oauth2/token",
}

In [0]:
source      = f"abfss://{container}@{storage_account}.dfs.core.windows.net/"
mount_point = f"/mnt/{container}"

try:
    dbutils.fs.mount(source=source, mount_point=mount_point, extra_configs=configs)
    display(dbutils.fs.ls(mount_point))
except Exception as e:
    print("Mount failed (expected — mounts disabled in this UC-first workspace):")
    print(e)

### Legacy SPN mount - outcome & migration

### The mount failed on purpose
`dbutils.fs.mount()` is a legacy way to attach storage to the workspace. Our GP1 cluster runs in
Unity Catalog mode, which blocks these old commands

### Migration
Start to access the same storage through a Unity Catalog external locatioon on a shared storage credential, reading via `abfss://.../` and external volumes instead of a `/mnt/...` path. 

### Why the scope is Databricks-backed, not Key Vault-backed
A KV-backed scope needs the Key Vault in "Access Policy" permission mode. My Stage 1 vault was created
with the RBAC model, and the permission model can't be changed after creation - so a KV-backed scope
would require a new Access-Policy vault. For this lab a Databricks-backed scope is equivalent:
`dbutils.secrets.get(...)` works identically.